# 文章をやさしい日本語に書き換える (テキスト平易化)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/25_summarization/text_simplification.ipynb)

NLP クックブック 25-09 ／ 要約と平易化

難しい文章を、子どもや日本語を学んでいる人にも読める、やさしい文章に書き換えたい。本当にやさしくなったか、意味が保たれているかも確かめたい。

In [1]:
!pip install -q fugashi unidic-lite regex "wordfreq[cjk]"

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def chat(messages, max_new_tokens=200):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

RULES = ("次の文章を「やさしい日本語」に書き換えてください。\n"
         "- 1 文を短くする (1 文に 1 つのことだけを書く)\n- 難しい言葉は、簡単な言葉に言い換える\n- 敬語や受け身、二重否定を使わない\n"
         "- 文末は「です」「ます」にする\n- 大事な情報 (日時、場所、してほしいこと) は残す\n- 書き換えた文章だけを出力する")
SHOT_IN = "台風の接近に伴い、本日午後より公共交通機関に大幅な遅延が生じる見込みであるため、不要不急の外出は控えられたい。"
SHOT_OUT = "台風が近づいています。今日の午後から、電車やバスがとても遅れます。大事な用事がない人は、外に出ないでください。"

def simplify(text):
    return chat([{"role": "user", "content": f"{RULES}\n\n{SHOT_IN}"}, {"role": "assistant", "content": SHOT_OUT},
                 {"role": "user", "content": f"{RULES}\n\n{text}"}])

texts = ["本日未明に発生した地震の影響により、当該地域では断水が継続しており、復旧の見通しは立っていないことから、給水所の利用を検討されたい。",
         "転入届は、新住所に居住を開始した日から 14 日以内に、本人確認書類を持参のうえ、市役所の窓口にて手続きを行う必要がある。",
         "発熱や咳などの症状が認められる場合には、医療機関を受診する前に、あらかじめ電話で相談することが推奨される。"]
simplified = [simplify(t) for t in texts]
for src, out in zip(texts, simplified):
    print(f"原文: {src}\n平易: {out}\n")

原文: 本日未明に発生した地震の影響により、当該地域では断水が継続しており、復旧の見通しは立っていないことから、給水所の利用を検討されたい。
平易: 今朝早くに起きた地震の影響で、ある地域では水が止まり続けています。まだ復旧する目処は立っていません。そのため、給水所を利用することをおすすめします。

原文: 転入届は、新住所に居住を開始した日から 14 日以内に、本人確認書類を持参のうえ、市役所の窓口にて手続きを行う必要がある。
平易: 転入届は、新しい住所に住み始めたら、14日以内に市役所に行って手続きをしてください。本人確認ができる書類も持っていってください。

原文: 発熱や咳などの症状が認められる場合には、医療機関を受診する前に、あらかじめ電話で相談することが推奨される。
平易: 熱があったり、咳が出たりする場合には、病院に行く前に、まず電話で相談することをおすすめします。



In [3]:
# 1. 本当にやさしくなったか: 文の長さ、漢字の割合、語彙の頻度 (よく使われる語ほど Zipf 値が高い) で比べる
import re
import pandas as pd
import regex
from fugashi import Tagger
from wordfreq import zipf_frequency

tagger = Tagger()

def difficulty(text):
    sents = [s for s in re.split(r"(?<=[。！？])", text) if s.strip()]
    words = [(w.surface, w.feature.pos1, w.feature.orthBase or w.surface) for s in sents for w in tagger(s) if w.feature.pos1 not in ("補助記号", "空白")]
    content = [base for _, pos, base in words if pos in ("名詞", "動詞", "形容詞", "形状詞", "副詞")]
    chars = regex.findall(r"\S", text)
    return {"文数": len(sents), "平均文長 (語)": round(len(words) / len(sents), 1),
            "漢字率": round(len(regex.findall(r"\p{Han}", text)) / len(chars), 2),
            "語の頻度 (平均)": round(sum(zipf_frequency(w, "ja") for w in content) / len(content), 2),
            "低頻度語の割合": round(sum(zipf_frequency(w, "ja") < 4 for w in content) / len(content), 2)}

rows = []
for i, (src, out) in enumerate(zip(texts, simplified), 1):
    rows += [{"文章": f"{i} 原文", **difficulty(src)}, {"文章": f"{i} 平易", **difficulty(out)}]
pd.DataFrame(rows)

,文章,文数,平均文長 (語),漢字率,語の頻度 (平均),低頻度語の割合
0,1 原文,1,40.0,0.45,4.94,0.18
1,1 平易,3,14.3,0.31,5.18,0.13
2,2 原文,1,35.0,0.54,4.94,0.14
3,2 平易,2,17.0,0.38,4.84,0.19
4,3 原文,1,29.0,0.40,5.04,0.24
5,3 平易,1,28.0,0.28,5.47,0.12


In [4]:
# 2. 意味が保たれているか: 平易化した文章の内容が原文から言えるか (含意) と、原文の要点が残っているか (逆向きの含意)
from transformers import AutoModelForSequenceClassification

nli_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
nli_tok = AutoTokenizer.from_pretrained(nli_name)
nli = AutoModelForSequenceClassification.from_pretrained(nli_name).eval()

def entail(premise, hypothesis):
    with torch.no_grad():
        return nli(**nli_tok(premise, hypothesis, return_tensors="pt", truncation=True)).logits.softmax(-1)[0, 0].item()

for src, out in zip(texts, simplified):
    out_sents = [s for s in re.split(r"(?<=[。！？])", out) if s.strip()]
    added = min(entail(src, s) for s in out_sents)          # 低いと、原文にないことを書き足している
    kept = entail(out, src)                                  # 低いと、原文の情報が落ちている
    print(f"付け足しなし {added:.2f} / 情報の保持 {kept:.2f} | {out[:40]}...")

付け足しなし 1.00 / 情報の保持 1.00 | 今朝早くに起きた地震の影響で、ある地域では水が止まり続けています。まだ復旧する目...
付け足しなし 1.00 / 情報の保持 1.00 | 転入届は、新しい住所に住み始めたら、14日以内に市役所に行って手続きをしてくださ...
付け足しなし 1.00 / 情報の保持 1.00 | 熱があったり、咳が出たりする場合には、病院に行く前に、まず電話で相談することをお...


- テキスト平易化は「意味を保ったまま、やさしく書き換える」タスク。評価も、やさしさ・意味の保持・自然さの 3 つを分けて見る。
- やさしい日本語には、1 文を短く、漢語より和語、敬語や受け身を避ける、といった具体的な指針がある。それを条件として列挙し、書き換えの例を 1 つ見せると従いやすい。
- 短く書き換える過程で、期限や持ち物のような大事な情報が落ちやすい。行政や防災の文章では、落ちていないかの確認が欠かせない。
- 参照となる平易文がある場合は SARI (追加・削除・保持の適否を測る指標) がよく使われる。日本語のデータには SNOW T15/T23 (やさしい日本語対訳コーパス)、JADES などがある。